# SoftLUT5 Debug — Isolating the AXI Hang

Run these cells **one by one** to find exactly where the board freezes.
Each cell is independent and tests one thing.

In [ ]:
# ---- Cell 0: Setup (always run first) ----
from pynq import Overlay, MMIO
import time

BITSTREAM = "/home/xilinx/softlut5_test/softlut5_test.bit"
ol = Overlay(BITSTREAM)
ip_name = [k for k in ol.ip_dict.keys() if "processing_system" not in k][0]
base_addr = ol.ip_dict[ip_name]['phys_addr']
mmio = MMIO(base_addr, 0x10000)
print(f"IP: {ip_name}, base: 0x{base_addr:08X}")
print("\u2705 Setup done")

In [ ]:
# ---- Cell 1: Read STATUS register (0x8000) ----
# If this hangs, the basic read path is broken.
print("Reading STATUS at 0x8000...")
val = mmio.read(0x8000)
print(f"STATUS = 0x{val:08X} (busy={val & 1})")
print("\u2705 Read STATUS passed")

In [ ]:
# ---- Cell 2: Read GATE_COUNT register (0x8004) ----
print("Reading GATE_COUNT at 0x8004...")
val = mmio.read(0x8004)
print(f"GATE_COUNT = {val} (expected: 1)")
print("\u2705 Read GATE_COUNT passed")

In [ ]:
# ---- Cell 3: Write to INPUT register (0x9000) ----
# This is an input register write — should get immediate BVALID.
# If this hangs, the write path itself is broken.
print("Writing 0x0000001F to input reg at 0x9000...")
mmio.write(0x9000, 0x0000001F)
print("\u2705 Input write passed")

In [ ]:
# ---- Cell 4: Read OUTPUT register (0x9004) ----
# This reads the LUT output. Should work regardless of gate programming.
print("Reading output at 0x9004...")
val = mmio.read(0x9004)
print(f"OUTPUT = 0x{val:08X} (LUT out bit 0 = {val & 1})")
print("\u2705 Read output passed")

In [ ]:
# ---- Cell 5: Write to a "don't care" address (0x4000) ----
# This address is NOT a gate and NOT an input register.
# The FSM should just ACK it immediately (b_valid_r <= 1).
# If this hangs, the AXI write handshake itself is broken.
print("Writing 0xDEADBEEF to unmapped address 0x4000...")
mmio.write(0x4000, 0xDEADBEEF)
print("\u2705 Don't-care write passed")

In [ ]:
# ---- Cell 6: Write to GATE address (0x0000) — THE CRASH TEST ----
# This triggers the 32-cycle shift FSM. BVALID is delayed ~33 cycles.
# If THIS hangs but Cell 5 doesn't, the problem is the delayed BVALID.
print("Writing 0x80000000 to gate address 0x0000...")
print("(This triggers the 32-cycle shift FSM)")
mmio.write(0x0000, 0x80000000)
print("\u2705 Gate write passed!")

In [ ]:
# ---- Cell 7: Read STATUS after gate write ----
# If Cell 6 passed but this hangs, the read-after-write is the issue.
print("Reading STATUS after gate write...")
val = mmio.read(0x8000)
print(f"STATUS = 0x{val:08X} (busy={val & 1})")
print("\u2705 Post-write status read passed")

In [ ]:
# ---- Cell 8: Full test — program + verify ----
# Only run this if all cells above passed.
print("Programming AND gate (INIT=0x80000000)...")
mmio.write(0x0000, 0x80000000)
# Wait a bit for shift to complete
time.sleep(0.001)
print("Verifying all 32 inputs...")
errors = 0
for i in range(32):
    mmio.write(0x9000, i & 0x1F)
    out = mmio.read(0x9004) & 0x1
    expected = 1 if i == 31 else 0
    if out != expected:
        print(f"  \u274c Input {i:05b} \u2192 got {out}, expected {expected}")
        errors += 1
if errors == 0:
    print("\u2705 AND gate: all 32 combinations PASSED!")
else:
    print(f"\u274c {errors}/32 FAILED")